In [3]:
from pathlib import Path
import sys
import json

# ============================================================
# User configuration
# ============================================================

STRUCTURE_PATH = Path(
    "/dtu/blackhole/00/222976/lead_sensoring/data/raw/rfd3_input_w6_scaffold-side-15A_5gpe_pb_motif_r5_scaffold-side-15A_1_model_0__b1_d1_model_2.cif"
)

OUTDIR = Path(
    "/dtu/blackhole/00/222976/lead_sensoring/work/vector_bild_only"
)

SCRIPTS_DIR = Path(
    "/dtu/blackhole/00/222976/lead_sensoring/scripts"
)

METAL = "PB"

# Cys SG atoms within this distance from Pb are considered nearby donors.
DONOR_CUTOFF = 3.4

# Arrow length in ChimeraX.
BILD_LENGTH = 12.0

# Open-space / ORI calculation parameters.
ORI_DISTANCES = [5.0, 8.0, 12.0, 15.0]
OPEN_SPACE_DIRECTIONS = 512
OPEN_SPACE_MAX_DISTANCE = 14.0
OPEN_SPACE_CONE_ANGLE = 45.0
CHEMICAL_WEIGHT = 0.0
OPEN_SPACE_TOP_K = 10

# Missing-residue diagnostic window.
GAP_WINDOW = 5


# ============================================================
# Imports from your existing project
# ============================================================

LIB_DIR = SCRIPTS_DIR / "lib"
sys.path.insert(0, str(LIB_DIR))

from constants import (
    DEFAULT_MAX_PB_S,
    DEFAULT_MIN_PB_S,
    DEFAULT_SIGMA_PB_S,
    DEFAULT_STRONG_EXTRA_DONOR_CUTOFF,
    DEFAULT_TARGET_PB_S,
    DEFAULT_WEAK_EXTRA_DONOR_CUTOFF,
)

from motif_scan import (
    analyze_one_pb_site,
    find_cys_sg_indices,
    find_metal_indices,
)

from open_space import add_open_space_and_ori_tokens

from structure_io import (
    find_observed_numbering_gaps,
    load_structure,
    parse_mmcif_missing_residues,
)

from visualization import write_pb_vector_bild


# ============================================================
# Main function
# ============================================================

def make_bild_for_each_pb_with_cys3(
    structure_path: Path,
    outdir: Path,
    metal: str = "PB",
    donor_cutoff: float = 3.4,
    bild_length: float = 12.0,
) -> list[Path]:
    """
    Create one ChimeraX BILD file per Pb site that has at least
    three nearby Cys SG donors.

    This is for visualization only.
    It does not apply RFD3 eligibility filters.
    """

    structure_path = structure_path.resolve()
    outdir = outdir.resolve()
    outdir.mkdir(parents=True, exist_ok=True)

    atoms = load_structure(structure_path)

    missing_residues = parse_mmcif_missing_residues(structure_path)
    observed_gaps = find_observed_numbering_gaps(atoms)

    metal_indices = find_metal_indices(atoms, metal)
    if not metal_indices:
        raise ValueError(f"No metal atoms found for metal={metal}")

    cys_sg_indices = find_cys_sg_indices(atoms)
    if not cys_sg_indices:
        raise ValueError("No Cys SG atoms found in the structure.")

    written_bilds = []
    site_diagnostics = []

    for metal_number, metal_index in enumerate(metal_indices, start=1):

        site = analyze_one_pb_site(
            atoms=atoms,
            metal_index=metal_index,
            metal_number=metal_number,
            cys_sg_indices=cys_sg_indices,
            donor_cutoff=donor_cutoff,
            target_pb_s=DEFAULT_TARGET_PB_S,
            sigma_pb_s=DEFAULT_SIGMA_PB_S,
            min_pb_s=DEFAULT_MIN_PB_S,
            max_pb_s=DEFAULT_MAX_PB_S,
            strong_extra_donor_cutoff=DEFAULT_STRONG_EXTRA_DONOR_CUTOFF,
            weak_extra_donor_cutoff=DEFAULT_WEAK_EXTRA_DONOR_CUTOFF,
            missing_residues=missing_residues,
            observed_gaps=observed_gaps,
            gap_window=GAP_WINDOW,
        )

        triad_donors = site.get("triad_donors", [])
        n_triad = len(triad_donors)

        site_diagnostics.append(site)

        print("=" * 70)
        print(f"Pb site {metal_number}")
        print(f"Site ID: {site.get('site_id')}")
        print(f"Geometry: {site.get('geometry_call')}")
        print(f"Score: {site.get('score')}")
        print(f"Triad donors found: {n_triad}")
        print([donor.get("residue") for donor in triad_donors])

        if n_triad < 3:
            print("Skipped: fewer than three nearby Cys SG donors.")
            continue

        # Add open-space vectors and ORI tokens.
        site = add_open_space_and_ori_tokens(
            atoms=atoms,
            selected_site=site,
            ori_distances=ORI_DISTANCES,
            n_directions=OPEN_SPACE_DIRECTIONS,
            max_distance=OPEN_SPACE_MAX_DISTANCE,
            cone_angle_deg=OPEN_SPACE_CONE_ANGLE,
            chemical_weight=CHEMICAL_WEIGHT,
            top_k=OPEN_SPACE_TOP_K,
        )

        bild_path = outdir / f"{structure_path.stem}_pb{metal_number}_vectors.bild"

        write_pb_vector_bild(
            selected_site=site,
            path=bild_path,
            length=bild_length,
        )

        written_bilds.append(bild_path)

        print("BILD written:")
        print(bild_path)

    diagnostics_path = outdir / f"{structure_path.stem}_all_pb_site_diagnostics.json"

    with diagnostics_path.open("w") as handle:
        json.dump(
            {
                "structure": str(structure_path),
                "metal": metal,
                "donor_cutoff": donor_cutoff,
                "missing_residues_from_mmcif": missing_residues,
                "observed_numbering_gaps": observed_gaps,
                "sites": site_diagnostics,
            },
            handle,
            indent=2,
        )

    print("=" * 70)
    print("Finished.")
    print(f"Number of Pb atoms found: {len(metal_indices)}")
    print(f"Number of BILD files written: {len(written_bilds)}")
    print("Diagnostics saved:")
    print(diagnostics_path)

    if written_bilds:
        print()
        print("Open in ChimeraX:")
        print(f"open {structure_path}")
        for bild_path in written_bilds:
            print(f"open {bild_path}")
    else:
        print()
        print("No BILD file was written because no Pb site had three nearby Cys SG donors.")

    return written_bilds


# ============================================================
# Run
# ============================================================

bild_paths = make_bild_for_each_pb_with_cys3(
    structure_path=STRUCTURE_PATH,
    outdir=OUTDIR,
    metal=METAL,
    donor_cutoff=DONOR_CUTOFF,
    bild_length=BILD_LENGTH,
)

Pb site 1
Site ID: PB_1
Geometry: distorted_pb_s3
Score: -476.097
Triad donors found: 3
['A71', 'A40', 'A80']
BILD written:
/dtu/blackhole/00/222976/lead_sensoring/work/vector_bild_only/rfd3_input_w6_scaffold-side-15A_5gpe_pb_motif_r5_scaffold-side-15A_1_model_0__b1_d1_model_2_pb1_vectors.bild
Finished.
Number of Pb atoms found: 1
Number of BILD files written: 1
Diagnostics saved:
/dtu/blackhole/00/222976/lead_sensoring/work/vector_bild_only/rfd3_input_w6_scaffold-side-15A_5gpe_pb_motif_r5_scaffold-side-15A_1_model_0__b1_d1_model_2_all_pb_site_diagnostics.json

Open in ChimeraX:
open /dtu/blackhole/00/222976/lead_sensoring/data/raw/rfd3_input_w6_scaffold-side-15A_5gpe_pb_motif_r5_scaffold-side-15A_1_model_0__b1_d1_model_2.cif
open /dtu/blackhole/00/222976/lead_sensoring/work/vector_bild_only/rfd3_input_w6_scaffold-side-15A_5gpe_pb_motif_r5_scaffold-side-15A_1_model_0__b1_d1_model_2_pb1_vectors.bild
